In [1]:
import xarray as xr
import climpred as cp
from xarray import DataArray

/scratch/tm70/ag9761/conda/envs/iceds/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
range_id = 3
dataset_name = "ukmo_604"
# Mask and scale is used for scaling factor
dataset = xr.open_dataset(f"/scratch/tm70/ag9761/iceds_datasets/{dataset_name}_{range_id}_new_forecast.nc", mask_and_scale=False, chunks="auto")
dataset = xr.decode_cf(dataset)

In [4]:
dataset

<xarray.Dataset> Size: 4GB
Dimensions:        (number: 62, indexing_time: 13, forecastMonth: 6,
                    latitude: 501, longitude: 421)
Coordinates:
  * number         (number) int64 496B 0 1 2 3 4 5 6 7 ... 55 56 57 58 59 60 61
  * indexing_time  (indexing_time) datetime64[ns] 104B 2025-05-01 ... 2026-05-01
  * forecastMonth  (forecastMonth) int64 48B 1 2 3 4 5 6
  * latitude       (latitude) float64 4kB -7.0 -7.05 -7.1 ... -31.9 -31.95 -32.0
  * longitude      (longitude) float64 3kB 135.0 135.1 135.1 ... 156.0 156.0
Data variables:
    tprate         (number, indexing_time, forecastMonth, latitude, longitude) float32 4GB dask.array<chunksize=(31, 6, 3, 265, 223), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             egrr
    GRIB_centreDescription:  U.K. Met Office - Exeter
    GRIB_subCentre:          98
    Conventions:             CF-1.7
    institution:             U.K. Met Office - Exeter
    history:                 2026-09-17T15:28 GRIB to CDM+CF via cfgrib-0.9.1...

In [5]:
def convert_units(da: DataArray, new_unit: str) -> DataArray:
        da = da.metpy.quantify()
        da = da.metpy.convert_units(new_unit)
        da = da.metpy.dequantify()
        return da

dataset["tprate"] = convert_units(dataset["tprate"], "mm/day")

In [6]:
# Make dataset ready for climpred
renamed_vars = ["indexing_time", "forecast_reference_time"]
for rv in renamed_vars:
    if rv in dataset.coords:
        dataset = dataset.rename({rv : "init"})
dataset = dataset.rename({"forecastMonth" : "lead", "number": "member", "tprate" : "pr"})
dataset["lead"].attrs["units"] = "months"
dataset = dataset.assign_coords(lead=dataset["lead"] - 1)

In [7]:
# Convert from init + lead config to valid_times for leads
model = cp.HindcastEnsemble(dataset).get_initialized()
dataset = cp.utils.convert_init_lead_to_valid_time_lead(model)
dataset

/scratch/tm70/ag9761/conda/envs/iceds/lib/python3.13/site-packages/climpred/classes.py:1003: UserWarning: HindcastEnsemble is chunked along dimensions ['member', 'init'] with more than one chunk. `HindcastEnsemble.chunks=Frozen({'member': (31, 31), 'init': (6, 6, 1), 'lead': (3, 3), 'latitude': (265, 236), 'longitude': (223, 198)})`.
You cannot call `HindcastEnsemble.verify` or `HindcastEnsemble.bootstrap` in combination with any of  ['member', 'init'] passed as `dim`. In order to do so, please rechunk ['member', 'init'] with `HindcastEnsemble.chunk({{dim:-1}}).verify(dim=dim).`
If you do not want to use dimensions  ['member', 'init'] in `HindcastEnsemble.verify(dim=dim)`, you can disregard this warning.
Consider chunking embarrassingly parallel dimensions such as ['latitude', 'longitude'] automatically, i.e. `HindcastEnsemble.chunk(latitude="auto").verify(...).
  warnings.warn(msg)


<xarray.Dataset> Size: 6GB
Dimensions:     (lead: 6, member: 62, valid_time: 18, latitude: 501,
                 longitude: 421)
Coordinates:
  * lead        (lead) int64 48B 0 1 2 3 4 5
  * member      (member) int64 496B 0 1 2 3 4 5 6 7 ... 54 55 56 57 58 59 60 61
  * valid_time  (valid_time) object 144B 2025-05-01 00:00:00 ... 2026-10-01 0...
    init        (lead, valid_time) object 864B 2025-05-01 00:00:00 ... 2026-0...
  * latitude    (latitude) float64 4kB -7.0 -7.05 -7.1 ... -31.9 -31.95 -32.0
  * longitude   (longitude) float64 3kB 135.0 135.1 135.1 ... 155.9 156.0 156.0
Data variables:
    pr          (lead, member, valid_time, latitude, longitude) float32 6GB dask.array<chunksize=(1, 31, 4, 265, 223), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             egrr
    GRIB_centreDescription:  U.K. Met Office - Exeter
    GRIB_subCentre:          98
    Conventions:             CF-1.7
    institution:             U.K. Met Office - Exeter

In [8]:
dataset = dataset.rename({"valid_time" : "time"})
dataset = dataset.drop_vars(["init"])

In [ ]:
if dataset_name == "ukmo_604":
    dataset = dataset.isel(member=slice(0, 8))

In [10]:
dataset

<xarray.Dataset> Size: 729MB
Dimensions:    (lead: 6, member: 8, time: 18, latitude: 501, longitude: 421)
Coordinates:
  * lead       (lead) int64 48B 0 1 2 3 4 5
  * member     (member) int64 64B 0 1 2 3 4 5 6 7
  * time       (time) object 144B 2025-05-01 00:00:00 ... 2026-10-01 00:00:00
  * latitude   (latitude) float64 4kB -7.0 -7.05 -7.1 ... -31.9 -31.95 -32.0
  * longitude  (longitude) float64 3kB 135.0 135.1 135.1 ... 155.9 156.0 156.0
Data variables:
    pr         (lead, member, time, latitude, longitude) float32 729MB dask.array<chunksize=(1, 8, 4, 265, 223), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             egrr
    GRIB_centreDescription:  U.K. Met Office - Exeter
    GRIB_subCentre:          98
    Conventions:             CF-1.7
    institution:             U.K. Met Office - Exeter

In [ ]:
# Prediction on ensemble median
dataset = dataset.median(dim='member')

In [8]:
dataset.to_netcdf(f"/scratch/tm70/ag9761/iceds_datasets/{dataset_name}_{range_id}_new_processed.nc")

/scratch/tm70/ag9761/conda/envs/iceds/lib/python3.13/site-packages/dask/_task_spec.py:767: RuntimeWarning: All-NaN slice encountered
  return self.func(*new_argspec, **kwargs)


In [ ]:
dataset

In [ ]:
import eccodes
from collections import defaultdict

pairs = defaultdict(set)
with open("/scratch/tm70/ag9761/_tmp_ukmo_604_2026_0.grib", "rb") as f:
    while (gid := eccodes.codes_grib_new_from_file(f)) is not None:
        pairs[eccodes.codes_get(gid, "dataDate")].add(eccodes.codes_get(gid, "number"))
        eccodes.codes_release(gid)

for date in sorted(pairs):
    print(date, sorted(pairs[date]))